In [1]:
from pathlib import Path

notebook_path = Path('/kaggle/input/notebooks/waithakafrancis/data-pipeline')
print("Contents of", notebook_path)
for item in notebook_path.iterdir():
    print("  ", item)

Contents of /kaggle/input/notebooks/waithakafrancis/data-pipeline
   /kaggle/input/notebooks/waithakafrancis/data-pipeline/__results__.html
   /kaggle/input/notebooks/waithakafrancis/data-pipeline/__notebook__.ipynb
   /kaggle/input/notebooks/waithakafrancis/data-pipeline/validation_reports
   /kaggle/input/notebooks/waithakafrancis/data-pipeline/__output__.json
   /kaggle/input/notebooks/waithakafrancis/data-pipeline/feature_store
   /kaggle/input/notebooks/waithakafrancis/data-pipeline/custom.css


In [2]:
# ── Load your data from the feature store ────────────────────────
import pandas as pd
from pathlib import Path

STORE = Path('/kaggle/input/notebooks/waithakafrancis/data-pipeline/feature_store')

# For CTGAN (Pipeline A)
a_train = pd.read_parquet(STORE / 'pipeline_a_train.parquet')

# For GANomaly (Pipeline B)
b_train = pd.read_parquet(STORE / 'pipeline_b_train.parquet')

# Holdout — used to evaluate BOTH models
holdout = pd.read_parquet(STORE / 'fraud_holdout.parquet')

print('CTGAN training shape  :', a_train.shape)
print('GANomaly training     :', b_train.shape)
print('Fraud holdout         :', holdout.shape)


CTGAN training shape  : (250170, 51)
GANomaly training     : (284315, 51)
Fraud holdout         : (492, 51)


In [3]:
# ── Verify your data loaded correctly ────────────────────────────
# CTGAN: should see ~9% fraud rate (post-SMOTE)
fraud_rate_a = a_train['Class'].mean()
print(f'Pipeline A fraud rate: {fraud_rate_a:.2%}')   # expect ~9%

# GANomaly: must have 0% fraud
fraud_rate_b = b_train['Class'].mean()
print(f'Pipeline B fraud rate: {fraud_rate_b:.2%}')   # expect 0.00%

# Feature count: should be 51
print(f'Features: {a_train.shape[1]}')               # expect 51

# For CTGAN — identify categorical columns
categorical_cols = ['Class']
continuous_cols  = [c for c in a_train.columns if c not in categorical_cols]
print(f'Continuous: {len(continuous_cols)} | Categorical: {len(categorical_cols)}')


Pipeline A fraud rate: 9.09%
Pipeline B fraud rate: 0.00%
Features: 51
Continuous: 50 | Categorical: 1


In [4]:
!pip install ctgan --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.9 MB/s eta 0:00:00a 0:00:01


In [5]:
import time
import json
import warnings
import dataclasses
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List

import wandb
import lightgbm as lgb
from ctgan import CTGAN
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, classification_report
)
from scipy.spatial.distance import jensenshannon
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)

print('Dependencies loaded.')

Dependencies loaded.


In [6]:
wandb.login(key='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')


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kkipngenokoech (kkipngenokoech-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [17]:
STORE = Path('/kaggle/input/notebooks/waithakafrancis/data-pipeline/feature_store')

a_train  = pd.read_parquet(STORE / 'pipeline_a_train.parquet')
a_val    = pd.read_parquet(STORE / 'pipeline_a_val.parquet')
holdout  = pd.read_parquet(STORE / 'fraud_holdout.parquet')

print(f'Pipeline A train shape : {a_train.shape}')
print(f'Pipeline A val shape   : {a_val.shape}')
print(f'Fraud holdout shape    : {holdout.shape}')
print(f'Pipeline A fraud rate  : {a_train["Class"].mean():.2%}')  # expect ~9%
print(f'Feature count          : {a_train.shape[1]}')             # expect 51

Pipeline A train shape : (250170, 51)
Pipeline A val shape   : (56962, 51)
Fraud holdout shape    : (492, 51)
Pipeline A fraud rate  : 9.09%
Feature count          : 51


In [7]:
@dataclass
class CTGANConfig:
    # ── Data ──────────────────────────────────────────────────────────────
    target_col: str = 'Class'
    n_features: int = 51          # set dynamically below

    # ── CTGAN architecture ────────────────────────────────────────────────
    embedding_dim: int = 128
    generator_dim: tuple = (256, 256)
    discriminator_dim: tuple = (256, 256)
    generator_lr: float = 2e-4
    generator_decay: float = 1e-6
    discriminator_lr: float = 2e-4
    discriminator_decay: float = 1e-6
    batch_size: int = 500
    discriminator_steps: int = 1
    log_frequency: bool = True
    pac: int = 10

    # ── Training ──────────────────────────────────────────────────────────
    n_epochs: int = 300

    # ── LightGBM ──────────────────────────────────────────────────────────
    lgb_num_leaves: int = 63
    lgb_learning_rate: float = 0.05
    lgb_n_estimators: int = 500
    lgb_min_child_samples: int = 20
    lgb_subsample: float = 0.8
    lgb_colsample_bytree: float = 0.8
    lgb_reg_alpha: float = 0.1
    lgb_reg_lambda: float = 0.1
    lgb_early_stopping_rounds: int = 50

    # ── Synthetic generation ──────────────────────────────────────────────
    n_synthetic_fraud: int = 5000

    # ── Quality gate thresholds ───────────────────────────────────────────
    mmd_threshold: float = 0.05
    jsd_threshold: float = 0.1
    novelty_min_distance: float = 0.01
    disc_auc_threshold: float = 0.75


cfg = CTGANConfig()
cfg.n_features = a_train.shape[1] - 1   # exclude target
print(f'Config: embedding_dim={cfg.embedding_dim}, n_epochs={cfg.n_epochs}, '
      f'batch_size={cfg.batch_size}, n_features={cfg.n_features}')

Config: embedding_dim=128, n_epochs=300, batch_size=500, n_features=50


In [8]:
run = wandb.init(
    project = 'PRINCIPLES AND ENGINEERING APPLICATIONS OF AI',
    name    = 'fm-ctgan-pipeline-a-run-1',
    tags    = ['CTGAN', 'Pipeline-A', 'LightGBM'],
    config  = asdict(cfg),         # logs every param in CTGANConfig
)

# Sync config back so wandb sweeps can override values
wcfg = wandb.config
cfg.embedding_dim           = wcfg.embedding_dim
cfg.batch_size              = wcfg.batch_size
cfg.n_epochs                = wcfg.n_epochs
cfg.generator_lr            = wcfg.generator_lr
cfg.discriminator_lr        = wcfg.discriminator_lr
cfg.discriminator_steps     = wcfg.discriminator_steps
cfg.pac                     = wcfg.pac
cfg.n_synthetic_fraud       = wcfg.n_synthetic_fraud
cfg.lgb_num_leaves          = wcfg.lgb_num_leaves
cfg.lgb_learning_rate       = wcfg.lgb_learning_rate
cfg.lgb_n_estimators        = wcfg.lgb_n_estimators
cfg.lgb_subsample           = wcfg.lgb_subsample
cfg.lgb_colsample_bytree    = wcfg.lgb_colsample_bytree
cfg.lgb_reg_alpha           = wcfg.lgb_reg_alpha
cfg.lgb_reg_lambda          = wcfg.lgb_reg_lambda

print('W&B run initialised:', run.name)
print('W&B project URL    :', run.url)

W&B run initialised: fm-ctgan-pipeline-a-run-1
W&B project URL    : https://wandb.ai/kkipngenokoech-carnegie-mellon-university/PRINCIPLES%20AND%20ENGINEERING%20APPLICATIONS%20OF%20AI/runs/ezoohuvo


In [9]:
FEATURE_COLS = [c for c in a_train.columns if c != cfg.target_col]
CATEGORICAL_COLS = ['Class']

# CTGAN trains on the full feature set (including engineered features)
# because unlike the T-GAN, it handles mixed types natively.
# We train only on fraud rows so the generator learns the fraud manifold.
fraud_train_ctgan = a_train[a_train[cfg.target_col] == 1].reset_index(drop=True)

print(f'Fraud records for CTGAN training : {len(fraud_train_ctgan)}')
print(f'CTGAN training features          : {len(FEATURE_COLS)} continuous + '
      f'{len(CATEGORICAL_COLS)} categorical')
print(f'Fraud Class distribution         :\n{fraud_train_ctgan["Class"].value_counts()}')

Fraud records for CTGAN training : 22742
CTGAN training features          : 50 continuous + 1 categorical
Fraud Class distribution         :
Class
1    22742
Name: count, dtype: int64


In [10]:
# ── Step 1: Initialise CTGAN ──────────────────────────────────────────────

ctgan_model = CTGAN(
    embedding_dim        = cfg.embedding_dim,
    generator_dim        = cfg.generator_dim,
    discriminator_dim    = cfg.discriminator_dim,
    generator_lr         = cfg.generator_lr,
    generator_decay      = cfg.generator_decay,
    discriminator_lr     = cfg.discriminator_lr,
    discriminator_decay  = cfg.discriminator_decay,
    batch_size           = cfg.batch_size,
    discriminator_steps  = cfg.discriminator_steps,
    log_frequency        = cfg.log_frequency,
    pac                  = cfg.pac,
    epochs               = cfg.n_epochs,
    verbose              = True,           # prints per-epoch losses
)

print('CTGAN model initialised.')
print(f'  Generator dims    : {cfg.generator_dim}')
print(f'  Discriminator dims: {cfg.discriminator_dim}')
print(f'  Embedding dim     : {cfg.embedding_dim}')

CTGAN model initialised.
  Generator dims    : (256, 256)
  Discriminator dims: (256, 256)
  Embedding dim     : 128


In [11]:
# ── Step 2: Training Loop ─────────────────────────────────────────────────
# CTGAN's .fit() runs internally — we wrap it with epoch-level W&B hooks
# by subclassing and overriding _train_epoch(), mirroring your teammate's
# per-epoch wandb.log() pattern.

from ctgan.synthesizers.ctgan import CTGAN as _BaseCTGAN
import torch

class LoggedCTGAN(_BaseCTGAN):
    """CTGAN with per-epoch W&B logging — matches T-GAN train loop style."""

    def fit(self, train_data, discrete_columns=()):
        # Internal setup (copies CTGAN's own _fit preamble)
        self._validate_discrete_columns(train_data, discrete_columns)
        self._transformer = self._build_transformer()
        self._transformer.fit(train_data, discrete_columns)

        train_data = self._transformer.transform(train_data)

        self._data_sampler = self._build_data_sampler(
            train_data, discrete_columns
        )

        data_dim = self._transformer.output_dimensions

        self._generator     = self._build_generator(data_dim)
        self._discriminator = self._build_discriminator(data_dim)

        self._optimizer_G = torch.optim.Adam(
            self._generator.parameters(),
            lr=self._generator_lr, betas=(0.5, 0.9),
            weight_decay=self._generator_decay
        )
        self._optimizer_D = torch.optim.Adam(
            self._discriminator.parameters(),
            lr=self._discriminator_lr, betas=(0.5, 0.9),
            weight_decay=self._discriminator_decay
        )

        history = {'g_loss': [], 'd_loss': []}

        steps_per_epoch = max(len(train_data) // self._batch_size, 1)

        for epoch in range(self._epochs):
            g_losses_ep, d_losses_ep = [], []

            for _ in range(steps_per_epoch):
                # ── Discriminator steps ──────────────────────────────────
                for _ in range(self._discriminator_steps):
                    d_loss = self._train_discriminator(train_data)
                    d_losses_ep.append(d_loss)

                # ── Generator step ───────────────────────────────────────
                g_loss = self._train_generator(train_data)
                g_losses_ep.append(g_loss)

            mean_g = float(np.mean(g_losses_ep))
            mean_d = float(np.mean(d_losses_ep))
            history['g_loss'].append(mean_g)
            history['d_loss'].append(mean_d)

            # ── W&B: log every epoch (mirrors T-GAN pattern) ─────────────
            wandb.log({
                'train/epoch'  : epoch + 1,
                'train/g_loss' : mean_g,
                'train/d_loss' : mean_d,
            }, step=epoch + 1)

            if (epoch + 1) % 10 == 0:
                print(
                    f'Epoch [{epoch+1:3d}/{self._epochs}] '
                    f'G: {mean_g:+.4f} | D: {mean_d:+.4f}'
                )

        return history

In [12]:
# ── Simpler alternative if the subclass above raises AttributeError ───────

print('Starting CTGAN training...')
t0 = time.time()

ctgan_model.fit(
    fraud_train_ctgan,
    discrete_columns=CATEGORICAL_COLS
)

train_time_min = (time.time() - t0) / 60
print(f'Training complete in {train_time_min:.1f} min')

# Log summary training metrics (CTGAN stores loss history internally)
# Access via ctgan_model.loss_values if available in your version
if hasattr(ctgan_model, 'loss_values'):
    loss_df = ctgan_model.loss_values   # DataFrame: Epoch, Generator Loss, Discriminator Loss
    for _, row in loss_df.iterrows():
        wandb.log({
            'train/epoch'  : int(row['Epoch']),
            'train/g_loss' : float(row['Generator Loss']),
            'train/d_loss' : float(row['Discriminator Loss']),
        }, step=int(row['Epoch']))
    print(f'Logged {len(loss_df)} epochs to W&B.')
else:
    wandb.log({'train/train_time_min': train_time_min})
    print('loss_values not available in this CTGAN version — logged runtime only.')

Starting CTGAN training...


Gen. (+00.25) | Discrim. (-00.37): 100%|██████████| 300/300 [13:02<00:00,  2.61s/it]

Training complete in 15.1 min
Logged 300 epochs to W&B.


## Synthetic Data Generation


In [13]:
# ── Step 3: Generate Synthetic Fraud Samples ──────────────────────────────

print(f'Generating {cfg.n_synthetic_fraud} synthetic fraud samples...')

synthetic_raw = ctgan_model.sample(cfg.n_synthetic_fraud)

# CTGAN may generate non-integer Class values — force to 1
synthetic_raw['Class'] = 1

# Reorder columns to match training data
synthetic_fraud = synthetic_raw[a_train.columns].copy()

print(f'Synthetic fraud shape : {synthetic_fraud.shape}')
print(f'Synthetic Class dist  :\n{synthetic_fraud["Class"].value_counts()}')
print(f'Synthetic Amount range: {synthetic_fraud["Amount"].min():.2f} '
      f'to {synthetic_fraud["Amount"].max():.2f}')

# Log generation summary to W&B
syn_stats = synthetic_fraud[FEATURE_COLS].describe()
wandb.log({
    'synthetic/n_samples'    : len(synthetic_fraud),
    'synthetic/amount_mean'  : float(syn_stats.loc['mean', 'Amount']),
    'synthetic/amount_std'   : float(syn_stats.loc['std',  'Amount']),
})

Generating 5000 synthetic fraud samples...
Synthetic fraud shape : (5000, 51)
Synthetic Class dist  :
Class
1    5000
Name: count, dtype: int64
Synthetic Amount range: -0.37 to 20.60


### Synthetic Quality Gates (matches other experimenrt exactly)


In [14]:
# ── Step 4: Synthetic Quality Gates ───────────────────────────────────────
# Functions are identical to other experiments — same thresholds, same metrics.

def _compute_mmd(real, synthetic, gamma=1.0):
    cap = 500
    rng = np.random.default_rng(GLOBAL_SEED)
    r = real[rng.choice(len(real), min(cap, len(real)), replace=False)]
    s = synthetic[rng.choice(len(synthetic), min(cap, len(synthetic)), replace=False)]
    def rbf(A, B):
        return np.exp(-gamma * np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)).mean()
    return float(rbf(r, r) - 2 * rbf(r, s) + rbf(s, s))


def _compute_jsd(real, synthetic, n_bins=50):
    scores = []
    for i in range(real.shape[1]):
        combined = np.concatenate([real[:, i], synthetic[:, i]])
        bins     = np.linspace(combined.min(), combined.max(), n_bins + 1)
        r_prob   = np.histogram(real[:, i],      bins=bins, density=True)[0]
        s_prob   = np.histogram(synthetic[:, i], bins=bins, density=True)[0]
        r_prob  /= r_prob.sum() + 1e-12
        s_prob  /= s_prob.sum() + 1e-12
        scores.append(float(jensenshannon(r_prob, s_prob)))
    return float(np.mean(scores))


def _novelty_check(real, synthetic, min_dist=0.01, cap=200):
    rng   = np.random.default_rng(GLOBAL_SEED)
    r     = real[rng.choice(len(real), min(cap, len(real)), replace=False)]
    s     = synthetic[rng.choice(len(synthetic), min(cap, len(synthetic)), replace=False)]
    dists = np.sqrt(np.sum((s[:, None, :] - r[None, :, :]) ** 2, axis=-1))
    return float(dists.mean()) >= min_dist, float(dists.mean())


def _discriminability_check(real, synthetic, auc_threshold=0.75):
    X    = np.vstack([real, synthetic])
    y    = np.array([0] * len(real) + [1] * len(synthetic))
    X_sc = StandardScaler().fit_transform(X)
    clf  = LogisticRegression(max_iter=500, random_state=GLOBAL_SEED, class_weight='balanced')
    clf.fit(X_sc, y)
    auc  = float(roc_auc_score(y, clf.predict_proba(X_sc)[:, 1]))
    return auc <= auc_threshold, auc


real_arr = holdout[FEATURE_COLS].values.astype(np.float64)
syn_arr  = synthetic_fraud[FEATURE_COLS].values.astype(np.float64)

mmd               = _compute_mmd(real_arr, syn_arr)
jsd               = _compute_jsd(real_arr, syn_arr)
nov_pass, nov     = _novelty_check(real_arr, syn_arr)
disc_pass, dauc   = _discriminability_check(real_arr, syn_arr)

mmd_pass  = mmd <= cfg.mmd_threshold
jsd_pass  = jsd <= cfg.jsd_threshold
overall   = all([mmd_pass, jsd_pass, nov_pass, disc_pass])

print(
    f'[Synthetic Quality Gates]\n'
    f'  MMD          : {mmd:.6f}  | Threshold <= {cfg.mmd_threshold}         | {"PASS" if mmd_pass  else "FAIL"}\n'
    f'  JSD (mean)   : {jsd:.6f}  | Threshold <= {cfg.jsd_threshold}           | {"PASS" if jsd_pass  else "FAIL"}\n'
    f'  Novelty dist : {nov:.6f}  | Threshold >= {cfg.novelty_min_distance}          | {"PASS" if nov_pass  else "FAIL"}\n'
    f'  Disc. AUC    : {dauc:.4f}  | Threshold <= {cfg.disc_auc_threshold}          | {"PASS" if disc_pass else "FAIL"}\n'
    f'  Overall      : {"PASS" if overall else "FAIL"}'
)

wandb.log({
    'quality_gates/mmd'              : mmd,
    'quality_gates/jsd_mean'         : jsd,
    'quality_gates/novelty_min_dist' : nov,
    'quality_gates/discriminator_auc': dauc,
    'quality_gates/mmd_pass'         : int(mmd_pass),
    'quality_gates/jsd_pass'         : int(jsd_pass),
    'quality_gates/novelty_pass'     : int(nov_pass),
    'quality_gates/discriminator_pass': int(disc_pass),
    'quality_gates/overall_pass'     : int(overall),
})

if not overall:
    print('WARNING: Quality gates failed. Results below are diagnostic only.')

[Synthetic Quality Gates]
  MMD          : 0.004181  | Threshold <= 0.05         | PASS
  JSD (mean)   : 0.264149  | Threshold <= 0.1           | FAIL
  Novelty dist : 60599.472338  | Threshold >= 0.01          | PASS
  Disc. AUC    : 0.9996  | Threshold <= 0.75          | FAIL
  Overall      : FAIL


In [18]:
# ── Augmented training set ────────────────────────────────────────────────
augmented_train = pd.concat(
    [a_train, synthetic_fraud[a_train.columns]], ignore_index=True
).sample(frac=1, random_state=GLOBAL_SEED).reset_index(drop=True)

X_train_lgb = augmented_train[FEATURE_COLS]
y_train_lgb = augmented_train['Class']
X_val_lgb   = a_val[FEATURE_COLS]
y_val_lgb   = a_val['Class']
X_holdout   = holdout[FEATURE_COLS]
y_holdout   = holdout['Class']

aug_fraud_rate = y_train_lgb.mean()
print(f'Augmented train: {augmented_train.shape} | Fraud rate: {aug_fraud_rate:.2%}')

wandb.log({
    'lgb/augmented_train_size' : len(augmented_train),
    'lgb/augmented_fraud_rate' : aug_fraud_rate,
    'lgb/scale_pos_weight'     : (y_train_lgb == 0).sum() / (y_train_lgb == 1).sum(),
})

Augmented train: (255170, 51) | Fraud rate: 10.87%


In [21]:
# ── LightGBM Classifier ───────────────────────────────────────────────────
lgb_params = {
    'objective'         : 'binary',
    'metric'            : ['binary_logloss', 'auc'],
    'boosting_type'     : 'gbdt',
    'num_leaves'        : cfg.lgb_num_leaves,
    'max_depth'         : -1,
    'learning_rate'     : cfg.lgb_learning_rate,
    'n_estimators'      : cfg.lgb_n_estimators,
    'min_child_samples' : cfg.lgb_min_child_samples,
    'subsample'         : cfg.lgb_subsample,
    'colsample_bytree'  : cfg.lgb_colsample_bytree,
    'reg_alpha'         : cfg.lgb_reg_alpha,
    'reg_lambda'        : cfg.lgb_reg_lambda,
    'scale_pos_weight'  : (y_train_lgb == 0).sum() / (y_train_lgb == 1).sum(),
    'random_state'      : GLOBAL_SEED,
    'n_jobs'            : -1,
    'verbose'           : -1,
}

dtrain = lgb.Dataset(X_train_lgb, label=y_train_lgb)
dval   = lgb.Dataset(X_val_lgb,   label=y_val_lgb, reference=dtrain)

print('Training LightGBM...')
t0 = time.time()

lgb_model = lgb.train(
    lgb_params, dtrain,
    num_boost_round = cfg.lgb_n_estimators,
    valid_sets      = [dtrain, dval],
    valid_names     = ['train', 'val'],
    callbacks       = [
        lgb.early_stopping(stopping_rounds=cfg.lgb_early_stopping_rounds, verbose=True),
        lgb.log_evaluation(period=50),
    ]
)

lgb_time = time.time() - t0
print(f'LightGBM complete in {lgb_time:.1f}s | Best iteration: {lgb_model.best_iteration}')


Training LightGBM...
Training until validation scores don't improve for 50 rounds
[50]	train's binary_logloss: 0.0221966	train's auc: 0.999863	val's binary_logloss: 0.0151797	val's auc: 0.984652
[100]	train's binary_logloss: 0.00374376	train's auc: 0.999975	val's binary_logloss: 0.00415738	val's auc: 0.98381
Early stopping, best iteration is:
[55]	train's binary_logloss: 0.0182174	train's auc: 0.999866	val's binary_logloss: 0.012732	val's auc: 0.985102
LightGBM complete in 7.1s | Best iteration: 55


In [22]:
# ── Threshold search on validation set ───────────────────────────────────
val_probs = lgb_model.predict(X_val_lgb, num_iteration=lgb_model.best_iteration)
best_thresh, best_f1 = 0.5, 0.0
thresh_rows = []

for thresh in np.arange(0.05, 0.95, 0.01):
    preds = (val_probs >= thresh).astype(int)
    f1_t  = f1_score(y_val_lgb, preds, zero_division=0)
    p     = precision_score(y_val_lgb, preds, zero_division=0)
    r     = recall_score(y_val_lgb, preds, zero_division=0)
    thresh_rows.append({'threshold': round(thresh, 2), 'f1': f1_t, 'precision': p, 'recall': r})
    if f1_t > best_f1:
        best_f1, best_thresh = f1_t, thresh

wandb.log({'eval/threshold_sweep': wandb.Table(dataframe=pd.DataFrame(thresh_rows))})
print(f'Best threshold: {best_thresh:.2f} (val F1={best_f1:.4f})')

# ── Holdout evaluation ────────────────────────────────────────────────────
normal_sample = a_val[a_val['Class'] == 0].sample(n=5000, random_state=GLOBAL_SEED)
eval_df       = pd.concat([holdout, normal_sample], ignore_index=True)
X_eval        = eval_df[FEATURE_COLS]
y_eval        = eval_df['Class']
print(f'Eval set: {len(eval_df)} records | Fraud: {y_eval.sum()} | Normal: {(y_eval==0).sum()}')

t_infer           = time.time()
holdout_probs     = lgb_model.predict(X_eval, num_iteration=lgb_model.best_iteration)
inference_time_ms = (time.time() - t_infer) * 1000
holdout_preds     = (holdout_probs >= best_thresh).astype(int)

f1        = f1_score(y_eval, holdout_preds, zero_division=0)
precision = precision_score(y_eval, holdout_preds, zero_division=0)
recall    = recall_score(y_eval, holdout_preds, zero_division=0)
auroc     = roc_auc_score(y_eval, holdout_probs)

print('\n' + '='*55)
print(' CTGAN + LightGBM — Holdout Evaluation Results')
print('='*55)
print(f' F1-Score  : {f1:.4f}')
print(f' Precision : {precision:.4f}')
print(f' Recall    : {recall:.4f}')
print(f' AUROC     : {auroc:.4f}')
print(f' Inference : {inference_time_ms:.2f} ms ({len(X_eval)} records)')
print(f' Threshold : {best_thresh:.2f}')
print('='*55)
print(classification_report(y_eval, holdout_preds, target_names=['Legit', 'Fraud']))

wandb.log({
    'holdout/f1'               : f1,
    'holdout/precision'        : precision,
    'holdout/recall'           : recall,
    'holdout/auroc'            : auroc,
    'holdout/inference_time_ms': inference_time_ms,
    'holdout/threshold'        : best_thresh,
    'holdout/n_records'        : len(X_eval),
})

Best threshold: 0.92 (val F1=0.8333)
Eval set: 5492 records | Fraud: 492 | Normal: 5000

 CTGAN + LightGBM — Holdout Evaluation Results
 F1-Score  : 0.9199
 Precision : 1.0000
 Recall    : 0.8516
 AUROC     : 0.9999
 Inference : 10.94 ms (5492 records)
 Threshold : 0.92
              precision    recall  f1-score   support

       Legit       0.99      1.00      0.99      5000
       Fraud       1.00      0.85      0.92       492

    accuracy                           0.99      5492
   macro avg       0.99      0.93      0.96      5492
weighted avg       0.99      0.99      0.99      5492



In [23]:
results = {
    'model'               : 'CTGAN + LightGBM',
    'pipeline'            : 'A',
    'author'              : 'Francis Waithaka (fwaithak)',
    'f1'                  : round(f1, 4),
    'precision'           : round(precision, 4),
    'recall'              : round(recall, 4),
    'auroc'               : round(auroc, 4),
    'inference_time_ms'   : round(inference_time_ms, 2),
    'threshold'           : round(best_thresh, 2),
    'n_synthetic_fraud'   : cfg.n_synthetic_fraud,
    'quality_gates_passed': overall,
    'quality_gate_detail' : {
        'mmd': round(mmd, 6),  'mmd_pass': mmd_pass,
        'jsd': round(jsd, 6),  'jsd_pass': jsd_pass,
        'novelty_dist': round(nov, 6), 'novelty_pass': nov_pass,
        'disc_auc': round(dauc, 4),    'disc_pass': disc_pass,
    },
    'wandb_run_url': wandb.run.url,
}

out_path = Path('/kaggle/working/ctgan_results.json')
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

wandb.save(str(out_path))
print(f'Results saved to {out_path}')
print(json.dumps(results, indent=2))

wandb.run.summary['best_f1']    = f1
wandb.run.summary['best_auroc'] = auroc
wandb.run.summary['best_recall'] = recall

wandb.finish()
print('W&B run finished.')

wandb: WARNING Saving files without folders. If you want to preserve subdirectories pass base_path to wandb.save, i.e. wandb.save("/mnt/folder/file.h5", base_path="/mnt")
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Results saved to /kaggle/working/ctgan_results.json
{
  "model": "CTGAN + LightGBM",
  "pipeline": "A",
  "author": "Francis Waithaka (fwaithak)",
  "f1": 0.9199,
  "precision": 1.0,
  "recall": 0.8516,
  "auroc": 0.9999,
  "inference_time_ms": 10.94,
  "threshold": 0.92,
  "n_synthetic_fraud": 5000,
  "quality_gates_passed": false,
  "quality_gate_detail": {
    "mmd": 0.004181,
    "mmd_pass": true,
    "jsd": 0.264149,
    "jsd_pass": false,
    "novelty_dist": 60599.472338,
    "novelty_pass": true,
    "disc_auc": 0.9996,
    "disc_pass": false
  },
  "wandb_run_url": "https://wandb.ai/kkipngenokoech-carnegie-mellon-university/PRINCIPLES%20AND%20ENGINEERING%20APPLICATIONS%20OF%20AI/runs/ezoohuvo"
}


holdout/auroc,▁
holdout/f1,▁
holdout/inference_time_ms,▁
holdout/n_records,▁
holdout/precision,▁
holdout/recall,▁
holdout/threshold,▁
lgb/augmented_fraud_rate,▁
lgb/augmented_train_size,▁
lgb/scale_pos_weight,▁
+15,...


W&B run finished.
